# 01 — Data generation

## The business problem

A subscription business loses customers every month, and almost always finds out too late.
The question the retention team actually needs answered is:

> **Which customers are most likely to leave next quarter, why are they leaving, and what
> should we do about it?**

## Why a simulator instead of a public dataset

The well-known public churn datasets are *cross-sectional*: one row per customer, a churn
flag, and no history. You cannot build a time-aware model on them, you cannot demonstrate a
leakage-safe feature cutoff, and you cannot show behaviour changing in the run-up to a
cancellation — which is exactly what a real churn system is built to detect.

So this project simulates the process that produces churn:

```
latent customer traits -> monthly behaviour -> monthly churn hazard -> disengagement -> cancellation
```

Two properties matter for the credibility of everything downstream:

1. **Churn is never sampled from a label column.** Each month, every active customer draws a
   Bernoulli event whose probability is a logistic function of things an analyst can observe:
   contract type, chronic support pressure, usage decline, late payments, satisfaction,
   tenure, and the gap between what a customer pays and how much they actually use.
2. **An unobserved frailty term is hidden from the modelling table.** Real churn is partly
   driven by things the company cannot see — a competitor's sales call, a budget cut, a
   change of job. Keeping that variance out of the features caps achievable accuracy at a
   realistic level instead of the suspicious 0.99 AUC that leaky synthetic data produces.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#EAE6DF",
                     "axes.facecolor": "white", "figure.facecolor": "white",
                     "axes.edgecolor": "#C9C4BA", "axes.axisbelow": True,
                     "legend.frameon": False})
SLATE, CLAY, AMBER = "#335C67", "#9E2A2B", "#E09F3E"


In [2]:
from data_generator import HAZARD, PLAN_TIERS, CHURN_REASONS, DISENGAGE_DECAY, RECOVERY_RATE

print("Churn hazard coefficients (log-odds, applied monthly):")
for k, v in HAZARD.items():
    print(f"  {k:<18} {v:+.3f}")
print(f"\nDisengagement: usage decays x{DISENGAGE_DECAY} per month, "
      f"{RECOVERY_RATE:.0%} of disengaging customers recover unaided")

Churn hazard coefficients (log-odds, applied monthly):
  intercept          -4.050
  month_to_month     +1.150
  one_year           +0.450
  ticket_pressure    +0.620
  usage_decline      +1.050
  value_gap          +0.380
  late_payments      +0.600
  satisfaction       -0.500
  tenure             -0.026
  price_pressure     +0.550
  low_feature_use    +0.300
  new_customer       +0.550

Disengagement: usage decays x0.66 per month, 20% of disengaging customers recover unaided


## Two-stage churn

Customers rarely cancel out of nowhere. They **disengage first**: logins thin out, usage
falls, tickets rise, satisfaction slides. Only at the end of that run-up do they cancel.

The simulator reproduces this explicitly. The monthly hazard decides who *enters* a
disengagement phase; the phase itself lasts two to five months with visible deterioration;
at the end the customer cancels — unless they recover on their own, which about one in five
do. Those recoveries are important: they are the honest false positives that stop the
modelling problem from being trivially separable.

In [3]:
import data_generator as gen

demo = gen.generate(n_customers=5000, write=False, verbose=False)
for name, df in demo.items():
    print(f"{name:<12} {df.shape[0]:>8,} rows x {df.shape[1]:>2} cols")

customers       5,600 rows x 11 cols
plans              18 rows x  9 cols
dim_date           24 rows x  7 cols
usage          83,211 rows x  8 cols
payments       83,211 rows x  6 cols
support        24,426 rows x  7 cols
status          5,000 rows x  4 cols


In [4]:
demo["customers"].head(5)

,customer_id,customer_name,gender,age,city,state,signup_date,customer_segment,occupation,income_level,plan_id
0,CUS004720,Kristin Johnson,MALE,46.0,Vaughnfort,Nebraska,2022-08-04,Consumer,Publishing rights manager,High,PL14
1,CUS002262,Robert Bartlett,male,44.0,West Anthonyton,Hawaii,2021-02-05,Enterprise,Ophthalmologist,Very High,PL18
2,CUS003991,Tracey Sweeney,FEMALE,27.0,South Vanessa,Maine,2022-09-23,Enterprise,Consulting civil engineer,High,PL14
3,CUS002241,Jessica Fitzpatrick,FEMALE,66.0,South Bradley,New Hampshire,2024-03-03,Consumer,"Doctor, general practice",High,PL05
4,CUS004985,Marilyn Ochoa,Female,61.0,Yorkfort,Arkansas,2025-08-12,Consumer,Historic buildings inspector/conservation officer,High,PL05


In [5]:
demo["usage"].head(5)

,customer_id,date_id,login_frequency,support_tickets,usage_hours,data_consumption,feature_usage,satisfaction_score
0,CUS000001,20240801,55,0,51.70,146.190002,5,3.24
1,CUS000001,20240901,32,1,29.95,93.879997,4,3.54
2,CUS000001,20241001,46,0,38.33,112.089996,1,4.42
3,CUS000001,20241101,49,0,30.44,74.930000,5,4.46
4,CUS000001,20241201,30,0,27.65,73.540001,2,4.32


## The plan catalogue

Plans are a tier crossed with a contract term, which is what lets the model separate *price*
effects from *commitment* effects later on. Longer terms are discounted, exactly as they
would be commercially.

In [6]:
demo["plans"][["plan_id", "plan_name", "monthly_price", "contract_type", "data_limit"]].head(9)

,plan_id,plan_name,monthly_price,contract_type,data_limit
0,PL01,Starter Month-to-month,19.99,Month-to-month,50 GB
1,PL02,Starter One year,18.39,One year,50 GB
2,PL03,Starter Two year,16.99,Two year,50 GB
3,PL04,Basic Month-to-month,34.99,Month-to-month,150 GB
4,PL05,Basic One year,32.19,One year,150 GB
5,PL06,Basic Two year,29.74,Two year,150 GB
6,PL07,Standard Month-to-month,59.99,Month-to-month,500 GB
7,PL08,Standard One year,55.19,One year,500 GB
8,PL09,Standard Two year,50.99,Two year,500 GB


In [7]:
status = demo["status"]
print(f"Overall churn across the 24-month window: {status.churn_flag.mean():.1%}\n")
print(status.loc[status.churn_flag == 1, "cancellation_reason"]
      .value_counts(normalize=True).mul(100).round(1).to_string())

Overall churn across the 24-month window: 30.5%

cancellation_reason
Found competitor         26.9
Too expensive            19.0
Missing features         18.1
Poor customer service    16.4
Not using service        10.5
Technical problems        9.0


## The raw layer is deliberately dirty

The extracts are written the way a production export really arrives: duplicated rows from a
double load, five spellings of *Credit card*, impossible ages, refunds booked as negative
payments, instrumentation spikes, and missing readings. Cleaning these is notebook 02 —
they are not an accident.

In [8]:
raw_customers = pd.read_csv("../data/raw/customers.csv")
raw_payments = pd.read_csv("../data/raw/payments.csv", usecols=["payment_method", "payment_amount"])

print("Gender spellings:      ", sorted(raw_customers.gender.dropna().unique()))
print("Payment method labels: ", sorted(raw_payments.payment_method.dropna().unique()))
print(f"\nDuplicate customer_ids: {raw_customers.customer_id.duplicated().sum():,}")
print(f"Missing ages:           {raw_customers.age.isna().sum():,}")
print(f"Negative payments:      {(raw_payments.payment_amount < 0).sum():,}")

Gender spellings:       ['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'Other', 'female', 'male', 'other']
Payment method labels:  ['BANK TRANSFER', 'Bank transfer', 'CC', 'Credit Card', 'Credit card', 'DD', 'Direct debit', 'E-check', 'Electronic check', 'Mailed check', 'bank transfer', 'credit card', 'direct debit', 'electronic check', 'mailed check']

Duplicate customer_ids: 600
Missing ages:           1,226
Negative payments:      834


## The full dataset

The pipeline writes the real extracts to `data/raw/`. Everything downstream reads from
there.

In [9]:
import os
for f in sorted(os.listdir("../data/raw")):
    size = os.path.getsize(f"../data/raw/{f}") / 1e6
    print(f"  {f:<16} {size:>7.1f} MB")

  customers.csv        5.5 MB
  dim_date.csv         0.0 MB
  payments.csv        34.0 MB
  plans.csv            0.0 MB
  status.csv           1.1 MB
  support.csv         13.9 MB
  usage.csv           34.7 MB


**Next:** `02_data_cleaning.ipynb` repairs the extracts and engineers a modelling table
with a strict time cutoff between features and label.